## Análise de dados de exames médicos para previsão de doenças com modelo XGBoost

### Introdução

A epilepsia é um distúrbio do sistema nervoso central afetando cerca de 1,2% (3,4 milhões de pessoas) nos EUA e mais de 65 milhões em todo o mundo. Além disso, cerca de 1 em cada 26 pessoas desenvolverá epilepsia em algum momento da vida. Existem muitos tipos de convulsões, cada uma com sintomas diferentes, como perda de consciência, movimentos bruscos ou confusão.
Algumas convulsões são muito mais difíceis de detectar visualmente, os pacientes geralmente apresentam sintomas como não responder ou olhar sem expressão por um breve período de tempo. As convulsões podem ocorrer inesperadamente e podem resultar em lesões como queda, mordedura da língua ou perda do controle da urina ou fezes. 
Portanto, essas são algumas das razões pelas quais a detecção de convulsões é de extrema importância para pacientes sob supervisão médica que se suspeitem estar propensos a convulsões.Este projeto usará métodos de classificação binária para prever se um indivíduo está tendo uma convulsão em algum momento.

### Objetivo

Prever se um paciente está tendo uma convulsão ou não através de 178 leituras de EEG (Eletroencefalograma) por segundo.

### Avaliação

Como métrica de avaliação do modelo usaremos a AUC Score (Area Under The Curve Score), cujo valor vai de 1 a 100% e para esse problema o valor da métrica deve ser aproximadamente de 99%, uma vez que a previsão do modelo está relacionada a casos de vida ou morte. Usaremos a métrica calculada no dataset de validação.

### Dados

O conjunto de dados inclui 4097 leituras de eletroencefalograma (EEG) por paciente durante 23,5 segundos, com 500 pacientes no total. Os 4097 pontos de dados foram então divididos igualmente em 23 partes por paciente, cada parte é convertida em uma linha no conjunto de dados. Cada linha contém 178 leituras, que são transformadas em colunas; em outras palavras, existem 178 colunas que compõem um segundo das leituras de EEG. No total, existem 11.500 linhas e 179 colunas com a última coluna contendo o status do paciente, esteja ele tendo uma convulsão ou não.

Fonte: https://archive.ics.uci.edu/ml/datasets/Epileptic+Seizure+Recognition


## Pacotes Python Usados no Projeto

In [16]:
#!pip install -q -U watermark

In [17]:
#!pip install -q xgboost
#!pip install -q seaborn

In [18]:
# Import
import pickle
import sklearn as sk
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, roc_curve
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

## Carregando os Dados

In [19]:
# Carregando os dados
df = pd.read_csv("dados/dados_originais.csv")

In [20]:
# Shape
df.shape

(11500, 180)

In [21]:
# Visualizando alguns registros
df.head()

,Unnamed: 0,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X170,X171,X172,X173,X174,X175,X176,X177,X178,y
0,X21.V1.791,135,190,229,223,192,125,55,-9,-33,...,-17,-15,-31,-77,-103,-127,-116,-83,-51,4
1,X15.V1.924,386,382,356,331,320,315,307,272,244,...,164,150,146,152,157,156,154,143,129,1
2,X8.V1.1,-32,-39,-47,-37,-32,-36,-57,-73,-85,...,57,64,48,19,-12,-30,-35,-35,-36,5
3,X16.V1.60,-105,-101,-96,-92,-89,-95,-102,-100,-87,...,-82,-81,-80,-77,-85,-77,-72,-69,-65,5
4,X20.V1.54,-9,-65,-98,-102,-78,-48,-16,0,-21,...,4,2,-12,-32,-41,-65,-83,-89,-73,5


## Análise Exploratória e Definição da Variável Alvo

Vamos criar uma coluna chamada LABEL_VARIAVEL_TARGET em que 1 é quando um paciente está tendo uma convulsão e 0 é quando um paciente não está tendo uma convulsão.

Na última coluna, somente o valor 1 representa convulsão. Os demais valores não representam convulsão.

In [22]:
# Colocando True onde o valor for igual a 1 e False onde o valor for diferente.
df["LABEL_VARIAVEL_TARGET"] = df.y == 1

In [23]:
# Ajusta o tipo para int
df["LABEL_VARIAVEL_TARGET"] = df["LABEL_VARIAVEL_TARGET"].astype(int)

In [24]:
# Visualizando alguns registros
df["LABEL_VARIAVEL_TARGET"].head()

0    0
1    1
2    0
3    0
4    0
Name: LABEL_VARIAVEL_TARGET, dtype: int64

In [25]:
# A coluna original (y) que continha se um paciente está tendo uma convulsão será eliminada, pois era uma variável 
# categórica com 5 status diferentes.
df.pop('y')

0        4
1        1
2        5
3        5
4        5
        ..
11495    2
11496    1
11497    5
11498    3
11499    4
Name: y, Length: 11500, dtype: int64

In [26]:
# A primeira coluna será descartada devido à sua inutilidade em nosso modelo de aprendizado de máquina. 
# Não precisamos do ID
df.drop(df.columns[0], axis = 1, inplace = True)

In [27]:
# Visualizando alguns registros
df.head()

,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,...,X170,X171,X172,X173,X174,X175,X176,X177,X178,LABEL_VARIAVEL_TARGET
0,135,190,229,223,192,125,55,-9,-33,-38,...,-17,-15,-31,-77,-103,-127,-116,-83,-51,0
1,386,382,356,331,320,315,307,272,244,232,...,164,150,146,152,157,156,154,143,129,1
2,-32,-39,-47,-37,-32,-36,-57,-73,-85,-94,...,57,64,48,19,-12,-30,-35,-35,-36,0
3,-105,-101,-96,-92,-89,-95,-102,-100,-87,-79,...,-82,-81,-80,-77,-85,-77,-72,-69,-65,0
4,-9,-65,-98,-102,-78,-48,-16,0,-21,-59,...,4,2,-12,-32,-41,-65,-83,-89,-73,0


In [28]:
print("Número de colunas:", len(df.columns))

Número de colunas: 179


In [29]:
# Verificando valores ausentes
df.isna().sum().sum()

0

As colunas foram divididas para capturar a leitura do EEG em um ponto no tempo e todos os pontos no tempo (todas as 178 colunas) existem no mesmo segundo. Cada linha tem as 178 colunas com as medidas, mais a coluna indicando se as medidas são de convulsão ou não.

### Calculando a Prevalência da Classe Positiva

A prevalência é a porcentagem de amostras que tem a característica estamos tentando prever. Nesse cenário específico, significa que as pessoas que têm uma convulsão representam a classe positiva (ocorrência do evento), enquanto as que não sofrem convulsão representam a classe negativa (não ocorrência do evento). 

A taxa é calculada por (número de amostras positivas / número de amostras). Portanto, uma taxa de prevalência de 0,2 significa que 20% de nossa amostra são de pacientes que tiveram convulsão.

In [30]:
p_positivo =  sum(df["LABEL_VARIAVEL_TARGET"].values) / len(df["LABEL_VARIAVEL_TARGET"].values)

In [31]:
print("Prevalência da classe positiva: %.3f"%p_positivo)

Prevalência da classe positiva: 0.200


## Limpeza dos Dados

Para o conjunto de dados da epilepsia, existem 178 recursos (colunas), no entanto, uma vez que cada coluna representa um ponto de dados em um ponto específico no tempo e são todas as leituras de EEG, não há necessidade de realizar transformação adicional.

In [32]:
# Preparando o dataset somente com os dados de interesse
collist = df.columns.tolist()
cols_input = collist[0:178]
df_data = df[cols_input + ["LABEL_VARIAVEL_TARGET"]]

In [33]:
df_data.head(2)

,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,...,X170,X171,X172,X173,X174,X175,X176,X177,X178,LABEL_VARIAVEL_TARGET
0,135,190,229,223,192,125,55,-9,-33,-38,...,-17,-15,-31,-77,-103,-127,-116,-83,-51,0
1,386,382,356,331,320,315,307,272,244,232,...,164,150,146,152,157,156,154,143,129,1


In [34]:
# Checando se temos colunas duplicadas nos dados de entrada
dup_cols = set([x for x in cols_input if cols_input.count(x) > 1])
print(dup_cols)
assert len(dup_cols) == 0, "você duplicou colunas em cols_input"

set()


In [35]:
# Checando se temos colunas duplicadas no dataset final
cols_df_data = list(df_data.columns)
dup_cols = set([x for x in cols_df_data if cols_df_data.count(x) > 1])
print(dup_cols)
assert len(dup_cols) == 0,'você duplicou colunas em df_data'

set()


## Divisão dos Dados em Treino, Validação e Teste

Geralmente, podemos dividir o conjunto de dados em 50/25/25, 60/20/20, 70/15/15 como a divisão para amostras de treinamento / validação / teste; isso também depende de quantas amostras temos. Se tivermos um conjunto de dados extremamente grande (centenas de milhões de linhas), podemos usar uma divisão como 98/1/1. 

A divisão de treinamento é usada para treinar nosso algoritmo de aprendizado de máquina, por isso queremos usar a maioria de nosso conjunto de dados. O conjunto de dados de validação é usado para ajustar os hiperparâmetros e selecionar a abordagem de melhor desempenho. O conjunto de dados de teste é usado para testar a precisão do nosso modelo de aprendizado de máquina.

In [36]:
# Gerando amostras aleatórias dos dados
df_data = df_data.sample(n = len(df_data))

In [37]:
df_data

,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,...,X170,X171,X172,X173,X174,X175,X176,X177,X178,LABEL_VARIAVEL_TARGET
6686,-5,3,6,16,23,32,42,48,58,65,...,28,39,44,41,35,25,22,21,23,0
3461,3,3,6,12,15,23,25,26,22,23,...,27,19,14,10,13,19,20,27,26,0
2043,-94,-117,-140,-155,-154,-141,-117,-88,-72,-71,...,-17,7,34,57,71,79,73,54,41,0
8911,-21,-20,-18,-21,-25,-32,-37,-49,-52,-58,...,-81,-74,-71,-75,-77,-77,-75,-68,-60,0
11200,18,14,13,9,-6,-20,-31,-27,-10,5,...,8,56,94,105,88,62,39,42,46,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9030,31,31,27,24,24,35,38,37,35,34,...,-28,-19,-20,-15,-16,-22,-21,-20,-20,0
5486,-54,-50,-71,-74,-55,-19,-13,-15,-47,-50,...,-23,-60,-77,-98,-106,-100,-82,-75,-50,0
1679,48,49,51,56,63,71,76,71,59,41,...,-40,-41,-39,-32,-32,-32,-40,-50,-68,0
4000,-574,-546,-516,-499,-484,-470,-463,-448,-446,-431,...,-403,-400,-382,-366,-356,-399,-463,-541,-584,1


In [38]:
# Ajustando os índices do dataset
df_data = df_data.reset_index(drop = True)

In [39]:
# Gera um índice para a divisão (30% dos dados vão para validação/teste, ou seja, faremos divisão 70/15/15)
df_valid_teste = df_data.sample(frac = 0.3)
print("Tamanho da divisão de validação/teste: %.1f" % (len(df_valid_teste) / len(df_data)))

Tamanho da divisão de validação/teste: 0.3


In [40]:
# Dados de teste
df_teste = df_valid_teste.sample(frac = 0.5)
len(df_teste)

1725

In [41]:
# Dados se validação
df_valid = df_valid_teste.drop(df_teste.index)
len(df_valid)

1725

In [42]:
# Dados de treino
df_treino = df_data.drop(df_valid_teste.index)
len(df_treino)

8050

In [43]:
# Verifica se as amostras somadas totalizam o volume original dos dados
print('Todas as amostras (n = %d)'%len(df_data))
assert len(df_data) == (len(df_teste) + len(df_valid) + len(df_treino)), \
'O total não está igual'

Todas as amostras (n = 11500)


In [44]:
def calcula_prevalencia(y_actual):
    return sum(y_actual) / len(y_actual)

In [45]:
# Verifica a prevalência da classe positiva em cada subconjunto
print("Teste(n = %d): %.3f" % (len(df_teste), calcula_prevalencia(df_teste.LABEL_VARIAVEL_TARGET.values)))
print("Validação(n = %d): %.3f" % (len(df_valid), calcula_prevalencia(df_valid.LABEL_VARIAVEL_TARGET.values)))
print("Treino(n = %d): %.3f" % (len(df_treino), calcula_prevalencia(df_treino.LABEL_VARIAVEL_TARGET.values)))

Teste(n = 1725): 0.199
Validação(n = 1725): 0.200
Treino(n = 8050): 0.200


## Balanceamento de Classe

In [46]:
# Proporção de classes em treino antes do balanceamento de classe
df_treino.LABEL_VARIAVEL_TARGET.value_counts()

LABEL_VARIAVEL_TARGET
0    6439
1    1611
Name: count, dtype: int64

In [47]:
# Cria um índice
rows_pos = df_treino.LABEL_VARIAVEL_TARGET == 1

In [48]:
# Define valores positivos e negativos do índice
df_train_pos = df_treino.loc[rows_pos]
df_train_neg = df_treino.loc[~rows_pos]

In [49]:
# Valor mínimo
n = np.min([len(df_train_pos), len(df_train_neg)])

In [50]:
# Obtém valores aleatórios para o dataset de treino
df_treino_final = pd.concat([df_train_pos.sample(n = n, random_state = 64), 
                             df_train_neg.sample(n = n, random_state = 64)], 
                            axis = 0, 
                            ignore_index = True)

In [51]:
# Amostragem
df_treino_final = df_treino_final.sample(n = len(df_treino_final), random_state = 64).reset_index(drop = True)

In [52]:
df_treino_final.LABEL_VARIAVEL_TARGET.value_counts()

LABEL_VARIAVEL_TARGET
1    1611
0    1611
Name: count, dtype: int64

In [53]:
print('Balanceamento em Treino(n = %d): %.3f'%(len(df_treino_final), 
                                               calcula_prevalencia(df_treino_final.LABEL_VARIAVEL_TARGET.values)))

Balanceamento em Treino(n = 3222): 0.500


In [54]:
# Salvamos todos os datasets em disco no formato csv.
df_treino.to_csv('dados/dados_treino.csv', index = False)
df_treino_final.to_csv('dados/dados_treino_final.csv', index = False)
df_valid.to_csv('dados/dados_valid.csv', index = False)
df_teste.to_csv('dados/dados_teste.csv', index = False)

In [55]:
pickle.dump(cols_input, open('dados/cols_input.sav', 'wb'))

Criamos as Matrizes X e Y.

In [56]:
# X
X_treino = df_treino_final[cols_input].values
X_valid = df_valid[cols_input].values

In [57]:
# Y
y_treino = df_treino_final['LABEL_VARIAVEL_TARGET'].values
y_valid = df_valid['LABEL_VARIAVEL_TARGET'].values

In [58]:
# Print
print('Shape dos dados de treino:', X_treino.shape, y_treino.shape)
print('Shape dos dados de validação:', X_valid.shape, y_valid.shape)

Shape dos dados de treino: (3222, 178) (3222,)
Shape dos dados de validação: (1725, 178) (1725,)


In [59]:
X_treino

array([[ 350,  458,  450, ..., -368, -323, -255],
       [  87,   65,   43, ...,  108,  102,   92],
       [  42,   38,   44, ...,   -2,  -34,  -67],
       ...,
       [  55,    8,  -39, ...,   25,   13,    5],
       [-126, -147, -160, ...,   48,    4,   -7],
       [   9,   10,    7, ...,   10,    9,   13]])

## Padronização


In [60]:
# Cria o objeto
scaler = StandardScaler()

In [61]:
# Faz o fit
scaler.fit(X_treino)

,copy,True
,with_mean,True
,with_std,True


In [62]:
# Salva o objeto em disco e carrega para usamos adiante
scalerfile = 'dados/scaler.sav'

In [63]:
pickle.dump(scaler, open(scalerfile, 'wb'))
scaler = pickle.load(open(scalerfile, 'rb'))

In [64]:
# Aplica a normalização em nossas matrizes de dados
X_treino_tf = scaler.transform(X_treino)
X_valid_tf = scaler.transform(X_valid)

In [65]:
X_treino_tf

array([[ 1.4990646 ,  1.92943888,  1.92304125, ..., -1.43329613,
        -1.26284907, -0.98050208],
       [ 0.4202285 ,  0.32771947,  0.23805364, ...,  0.50224541,
         0.48114388,  0.43507375],
       [ 0.23563677,  0.21767768,  0.24219366, ...,  0.0549564 ,
        -0.07693386, -0.21356186],
       ...,
       [ 0.28896327,  0.09540902, -0.10142789, ...,  0.16474552,
         0.11593124,  0.08015993],
       [-0.45350568, -0.53631238, -0.60237015, ...,  0.25826959,
         0.07899963,  0.03120629],
       [ 0.1002695 ,  0.10356026,  0.08901297, ...,  0.10375157,
         0.09951719,  0.11279568]])

## Modelagem Preditiva

In [66]:
# Função para calcular a especificidade
def calc_specificity(y_actual, y_pred, thresh):
    return sum((y_pred < thresh) & (y_actual == 0)) / sum(y_actual ==0)

# Função para gerar relatório de métricas
def print_report(y_actual, y_pred, thresh):
    
    auc = roc_auc_score(y_actual, y_pred)
    
    accuracy = accuracy_score(y_actual, (y_pred > thresh))
    
    recall = recall_score(y_actual, (y_pred > thresh))
    
    precision = precision_score(y_actual, (y_pred > thresh))
    
    specificity = calc_specificity(y_actual, y_pred, thresh)
    
    print('AUC:%.3f'%auc)
    print('Acurácia:%.3f'%accuracy)
    print('Recall:%.3f'%recall)
    print('Precisão:%.3f'%precision)
    print('Especificidade:%.3f'%specificity)
    print(' ')
    
    return auc, accuracy, recall, precision, specificity 

In [67]:
thresh = 0.5

### Versão 1 - Modelo de Regressão Logistica

In [68]:
# Construção do modelo

# Cria o classificador (objeto)
lr = LogisticRegression(max_iter = 500, random_state = 142)


In [69]:
# Treina e cria o modelo
modelo_v1 = lr.fit(X_treino_tf, y_treino)

# Previsões 
y_train_preds = modelo_v1.predict_proba(X_treino_tf)[:,1]
y_valid_preds = modelo_v1.predict_proba(X_valid_tf)[:,1]

print('\nRegressão Logística\n')

print('Treinamento:\n')
v1_train_auc, v1_train_acc, v1_train_rec, v1_train_prec, v1_train_spec = print_report(y_treino, 
                                                                                      y_train_preds, 
                                                                                      thresh)

print('Validação:\n')
v1_valid_auc, v1_valid_acc, v1_valid_rec, v1_valid_prec, v1_valid_spec = print_report(y_valid, 
                                                                                      y_valid_preds, 
                                                                                      thresh)


Regressão Logística

Treinamento:

AUC:0.610
Acurácia:0.642
Recall:0.518
Precisão:0.688
Especificidade:0.765
 
Validação:

AUC:0.556
Acurácia:0.681
Recall:0.472
Precisão:0.307
Especificidade:0.733
 


### Versão 2 - Modelo Naive Bayes

O modelo Naive Bayes é um classificador probabilístico fundamentado no Teorema de Bayes, que opera sob a suposição "ingênua" de independência entre os preditores.

Em outras palavras, assume que a presença de uma característica particular em uma classe é independente da presença de qualquer outra característica. Essa suposição simplifica os cálculos, permitindo que o modelo seja treinado de maneira eficiente mesmo em grandes conjuntos de dados.

O modelo é utilizado para tarefas de classificação, onde o objetivo é prever a probabilidade de uma observação pertencer a uma das classes.

A base do Naive Bayes é o cálculo das probabilidades condicionais de cada classe dadas as características observadas, utilizando o Teorema de Bayes para atualizar as probabilidades a priori das classes com base nas evidências fornecidas pelos dados.

In [71]:
# Cria o classificador (objeto)
nb = GaussianNB()

In [72]:
# Treina e cria o modelo
modelo_v2 = nb.fit(X_treino_tf, y_treino)

# Previsões
y_train_preds = modelo_v2.predict_proba(X_treino_tf)[:,1]
y_valid_preds = modelo_v2.predict_proba(X_valid_tf)[:,1]

print('\nNaive Bayes\n')

print('Treinamento:\n')
v2_train_auc, v2_train_acc, v2_train_rec, v2_train_prec, v2_train_spec = print_report(y_treino, 
                                                                                     y_train_preds, 
                                                                                     thresh)

print('Validação:\n')
v2_valid_auc, v2_valid_acc, v2_valid_rec, v2_valid_prec, v2_valid_spec = print_report(y_valid, 
                                                                                      y_valid_preds, 
                                                                                      thresh)


Naive Bayes

Treinamento:

AUC:0.985
Acurácia:0.938
Recall:0.904
Precisão:0.969
Especificidade:0.971
 
Validação:

AUC:0.983
Acurácia:0.961
Recall:0.922
Precisão:0.886
Especificidade:0.970
 


### Versão 3 - Modelo XGBoost (Xtreme Gradient Boosting Classifier)

O XGBoost (Xtreme Gradient Boosting) é uma implementação avançada do algoritmo gradiente boosting que tem ganhado popularidade devido à sua eficiência e eficácia em competições de Ciência de Dados e aplicações práticas.

O princípio por trás do XGBoost é a otimização sequencial de árvores de decisão, onde cada nova árvore é construída para corrigir os erros residuais das árvores anteriores.

Diferentemente de outros métodos de boosting, o XGBoost utiliza um modelo formal de otimização para encontrar a melhor forma de combinar as previsões das árvores, minimizando uma função de perda que quantifica a diferença entre as previsões e os verdadeiros valores.

Além disso, incorpora técnicas de regularização (L1 e L2) para controlar a complexidade do modelo, ajudando a prevenir o overfitting e melhorando a capacidade de generalização do modelo.

In [73]:
# Cria o classificador
xgbc = XGBClassifier()

In [74]:
# Treina e cria o modelo
modelo_v3 = xgbc.fit(X_treino_tf, y_treino)

# Previsões
y_train_preds = modelo_v3.predict_proba(X_treino_tf)[:,1]
y_valid_preds = modelo_v3.predict_proba(X_valid_tf)[:,1]

print('\nXtreme Gradient Boosting Classifier\n')

print('Treinamento:\n')
v3_train_auc, v3_train_acc, v3_train_rec, v3_train_prec, v3_train_spec = print_report(y_treino, 
                                                                                   y_train_preds, 
                                                                                   thresh)

print('Validação:\n')
v3_valid_auc, v3_valid_acc, v3_valid_rec, v3_valid_prec, v3_valid_spec = print_report(y_valid, 
                                                                                     y_valid_preds, 
                                                                                     thresh)


Xtreme Gradient Boosting Classifier

Treinamento:

AUC:1.000
Acurácia:1.000
Recall:1.000
Precisão:1.000
Especificidade:1.000
 
Validação:

AUC:0.991
Acurácia:0.963
Recall:0.957
Precisão:0.871
Especificidade:0.964
 


## Validação Cruzada

Para realizar a validação cruzada no modeloXGBClassifier, pode-se usar a função `cross_val_score` da biblioteca `scikit-learn`. Esta função permite avaliar a eficácia do modelo usando diferentes divisões de treino/teste, o que ajuda a obter uma estimativa mais robusta do desempenho do modelo.

In [ ]:
# Cria o classificador
xgbc = XGBClassifier()

In [ ]:
# Configura a validação cruzada
# Usando 5 divisões e a métrica de área sob a curva ROC (AUC)
n_splits = 5

# Realiza a validação cruzada
cv_scores = cross_val_score(xgbc, X_treino_tf, y_treino, 
                            cv = n_splits, scoring = 'roc_auc')

# Exibe os resultados
print(f"Validação Cruzada com {n_splits} divisões")
print(f"Score AUC em Cada Divisão: {cv_scores}")
print(f"Média de Score AUC: {np.mean(cv_scores)}")

Validação Cruzada com 5 divisões
Score AUC em Cada Divisão: [0.99537527 0.99231775 0.99158019 0.99307511 0.99147409]
Média de Score AUC: 0.992764482173329


## Otimização de Hiperparâmetros

Para realizar a otimização de hiperparâmetros no seu modelo XGBoost, pode-se usar o `GridSearchCV` ou `RandomizedSearchCV` da biblioteca `scikit-learn.` Essas ferramentas permitem testar automaticamente várias combinações de hiperparâmetros e selecionar a melhor combinação com base no desempenho do modelo.

O processo com RandomizedSearchCV seria similar ao GridSearchCV, mas com a diferença de que ele testa um número fixo de combinações de hiperparâmetros escolhidas aleatoriamente, o que pode ser mais eficiente se o espaço de hiperparâmetros for muito grande.

In [78]:
%%time

# Define o espaço de hiperparâmetros para a otimização
param_grid = {
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 200, 300],
    'subsample': [0.7, 0.8, 0.9]
}

# Configura o GridSearchCV
grid_search = GridSearchCV(xgbc, param_grid, cv = 5, scoring = 'roc_auc', n_jobs = -1)

# Realiza a busca pelos melhores hiperparâmetros
grid_search.fit(X_treino_tf, y_treino)

# Melhores hiperparâmetros encontrados
best_params = grid_search.best_params_

# Treina o modelo com os melhores hiperparâmetros
modelo_v4 = grid_search.best_estimator_

# Previsões com o modelo otimizado
y_train_preds_optimized = modelo_v4.predict_proba(X_treino_tf)[:,1]
y_valid_preds_optimized = modelo_v4.predict_proba(X_valid_tf)[:,1]

CPU times: user 1min 55s, sys: 818 ms, total: 1min 56s
Wall time: 4min 17s


In [ ]:
# Avaliação do modelo otimizado
print('\nXtreme Gradient Boosting Classifier - Otimizado\n')
print('Melhores hiperparâmetros:\n', best_params)

print('\nTreinamento:\n')
v4_train_auc, v4_train_acc, v4_train_rec, v4_train_prec, v4_train_spec = print_report(y_treino, 
                                                                                          y_train_preds_optimized, 
                                                                                          thresh)

print('Validação:\n')
v4_valid_auc, v4_valid_acc, v4_valid_rec, v4_valid_prec, v4_valid_spec = print_report(y_valid, 
                                                                                          y_valid_preds_optimized, 
                                                                                          thresh)


Xtreme Gradient Boosting Classifier - Otimizado

Melhores hiperparâmetros: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 300, 'subsample': 0.9}

Treinamento:

AUC:1.000
Acurácia:1.000
Recall:1.000
Precisão:1.000
Especificidade:1.000
 
Validação:

AUC:0.992
Acurácia:0.966
Recall:0.954
Precisão:0.887
Especificidade:0.970
 


## Melhor Modelo

In [79]:
# Cria um dataframe com as métricas calculadas
df_results = pd.DataFrame({'classificador':['RL','RL','NB','NB','XGB','XGB','XGB_O','XGB_O'],
                           'data_set':['treino','validação'] * 4,
                           'auc':[v1_train_auc,
                                  v1_valid_auc,
                                  v2_train_auc,
                                  v2_valid_auc,
                                  v3_train_auc,
                                  v3_valid_auc,
                                  v4_train_auc,
                                  v4_valid_auc],
                           'accuracy':[v1_train_acc,
                                       v1_valid_acc,
                                       v2_train_acc,
                                       v2_valid_acc,
                                       v3_train_acc,
                                       v3_valid_acc,
                                       v4_train_acc,
                                       v4_valid_acc],
                           'recall':[v1_train_rec,
                                     v1_valid_rec,
                                     v2_train_rec,
                                     v2_valid_rec,
                                     v3_train_rec,
                                     v3_valid_rec,
                                     v4_train_rec,
                                     v4_valid_rec],
                           'precision':[v1_train_prec,
                                        v1_valid_prec,
                                        v2_train_prec,
                                        v2_valid_prec,
                                        v3_train_prec,
                                        v3_valid_prec,
                                        v4_train_prec,
                                        v4_valid_prec],
                           'specificity':[v1_train_spec,
                                          v1_valid_spec,
                                          v2_train_spec,
                                          v2_valid_spec,
                                          v3_train_spec,
                                          v3_valid_spec,
                                          v4_train_spec,
                                          v4_valid_spec]})

In [81]:
df_results[df_results['data_set'] == 'validação'].sort_values(by = 'auc', ascending = False)

,classificador,data_set,auc,accuracy,recall,precision,specificity
7,XGB_O,validação,0.992487,0.966377,0.953623,0.886792,0.969565
5,XGB,validação,0.991477,0.962899,0.956522,0.870712,0.964493
3,NB,validação,0.982699,0.960580,0.921739,0.885794,0.970290
1,RL,validação,0.556335,0.681159,0.472464,0.306968,0.733333


In [82]:
df_results[df_results['data_set'] == 'treino'].sort_values(by = 'auc', ascending = False)

,classificador,data_set,auc,accuracy,recall,precision,specificity
4,XGB,treino,1.000000,1.000000,1.000000,1.000000,1.000000
6,XGB_O,treino,1.000000,1.000000,1.000000,1.000000,1.000000
2,NB,treino,0.985103,0.937616,0.903786,0.969374,0.971446
0,RL,treino,0.610162,0.641527,0.517691,0.688119,0.765363


Considerando a métrica AUC, o melhor modelo é o **Classificador XGBoost Versão Otimizada** (versão 4).